# Principal Component Analysis (PCA)

Notebook นี้ใช้ Breast Cancer Wisconsin dataset เพื่อเรียนรู้การ standardize ข้อมูล เลือกจำนวน principal components และประเมินผลต่อ classification อย่างป้องกัน data leakage.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Load and Standardize Data

PCA ขับเคลื่อนด้วย variance ดังนั้น feature ที่มี scale ใหญ่สามารถครอบงำผลลัพธ์ได้. จึง standardize ทุก feature ให้มีค่าเฉลี่ย 0 และส่วนเบี่ยงเบนมาตรฐาน 1 ก่อนทำ PCA.

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
feature_names = cancer.feature_names
class_names = cancer.target_names

X_scaled = StandardScaler().fit_transform(X)

print('Feature matrix shape:', X.shape)
print('Classes:', dict(enumerate(class_names)))
pd.DataFrame(X, columns=feature_names).head()

## 2. Explained Variance and Scree Plot

PCA เรียง components จาก variance ที่อธิบายได้มากไปน้อย. Scree plot และ cumulative variance ช่วยตอบว่าเก็บกี่ components จึงสมเหตุสมผล.

In [ ]:
pca_full = PCA().fit(X_scaled)
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
components = np.arange(1, len(explained_variance) + 1)

components_90 = np.searchsorted(cumulative_variance, 0.90) + 1
components_95 = np.searchsorted(cumulative_variance, 0.95) + 1

print(f'Components for 90% variance: {components_90}')
print(f'Components for 95% variance: {components_95}')

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(components, explained_variance, color='#2563eb')
axes[0].set(title='Scree plot', xlabel='Principal component', ylabel='Explained variance ratio')

axes[1].plot(components, cumulative_variance, marker='o', markersize=3, color='#059669')
axes[1].axhline(0.90, linestyle='--', color='#d97706', label='90% target')
axes[1].axhline(0.95, linestyle='--', color='#dc2626', label='95% target')
axes[1].axvline(components_90, linestyle=':', color='#d97706')
axes[1].axvline(components_95, linestyle=':', color='#dc2626')
axes[1].set(title='Cumulative explained variance', xlabel='Components retained', ylabel='Cumulative variance', ylim=(0, 1.02))
axes[1].legend(frameon=False)

for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Visualize a 2D PCA Projection

การฉายเป็น 2D ช่วยสำรวจข้อมูล แต่ไม่ได้หมายความว่า 2 components จะดีที่สุดสำหรับ classification.

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
for class_label, class_name, color in zip(np.unique(y), class_names, ['#2563eb', '#dc2626']):
    mask = y == class_label
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], color=color, label=class_name, alpha=0.75)
plt.title(f'2D PCA projection ({pca_2d.explained_variance_ratio_.sum():.1%} variance retained)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(frameon=False)
plt.grid(alpha=0.2)
plt.show()

## 4. Evaluate PCA for Classification

เปรียบเทียบ Logistic Regression บน features เดิมกับ PCA หลายค่า $k$. วาง `StandardScaler` และ `PCA` ใน `Pipeline` เพื่อให้แต่ละ cross-validation fold fit transformations จาก training split ของตัวเอง.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
component_options = [2, 7, 10, 20]

baseline = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
])

score_rows = [{
    'representation': 'Original features',
    'components': X.shape[1],
    'mean_accuracy': cross_val_score(baseline, X, y, cv=folds).mean(),
    'std_accuracy': cross_val_score(baseline, X, y, cv=folds).std(),
}]

for component_count in component_options:
    pipeline = Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(n_components=component_count, random_state=RANDOM_STATE)),
        ('model', LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
    ])
    scores = cross_val_score(pipeline, X, y, cv=folds)
    score_rows.append({
        'representation': f'PCA with {component_count} components',
        'components': component_count,
        'mean_accuracy': scores.mean(),
        'std_accuracy': scores.std(),
    })

classification_results = pd.DataFrame(score_rows)
classification_results.round(3)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.errorbar(
    classification_results['components'],
    classification_results['mean_accuracy'],
    yerr=classification_results['std_accuracy'],
    marker='o',
    capsize=4,
    color='#2563eb',
)
plt.xlabel('Number of features or PCA components')
plt.ylabel('Cross-validation accuracy')
plt.title('Accuracy versus representation size')
plt.grid(alpha=0.25)
plt.show()

## Interpretation Checklist

- เลือก $k$ จาก explained variance และผลของ downstream task ร่วมกัน
- 2D PCA เหมาะกับการสำรวจ ไม่จำเป็นต้องเป็น input ที่ดีที่สุดของ classifier
- PCA สร้าง features ใหม่ จึงแลกความกระชับกับการตีความ feature เดิมโดยตรง
- Fit scaler และ PCA บน training data เท่านั้น เพื่อหลีกเลี่ยง data leakage